In [5]:
# PyPDFLoader đọc nội dung và số trang từ PDF.
from langchain_community.document_loaders import PyPDFLoader
# RecursiveCharacterTextSplitter chia văn bản dài thành các đoạn nhỏ.
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re


In [6]:
pdf_path = r'D:\langchain-study\data\Các bộ luật.pdf'
documents = PyPDFLoader(pdf_path).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(documents)


In [7]:
def search(query, k=3):
    article = re.search(r'\bđiều\s+(\d+)\b', query, flags=re.IGNORECASE)

    if article:
        # Chỉ giữ các đoạn có đúng số điều được hỏi.
        pattern = rf'\bđiều\s+{article.group(1)}\b'
        matches = [doc for doc in splits if re.search(pattern, doc.page_content, re.IGNORECASE)]
        if matches:
            # Chọn đoạn có nhiều nội dung nhất sau tiêu đề điều luật.
            return [max(matches, key=lambda doc: len(doc.page_content) - re.search(pattern, doc.page_content, re.IGNORECASE).end())]

    # Nếu không hỏi số điều: tách câu hỏi thành các từ khóa.
    keywords = set(re.findall(r'\w+', query.lower())) - {'là', 'gì', 'của', 'và'}

    def score(document):
        # Đếm số lần mỗi từ khóa xuất hiện trong một đoạn.
        words = re.findall(r'\w+', document.page_content.lower())
        return sum(words.count(keyword) for keyword in keywords)

    # Sắp xếp theo điểm giảm dần và chỉ lấy k kết quả đầu.
    return sorted(splits, key=score, reverse=True)[:k]


In [8]:
# Câu hỏi bạn muốn tìm trong PDF.
question = 'chương 1 là gì?'
# Gọi hàm tìm kiếm ở cell trên.
results = search(question)

# enumerate đánh số kết quả bắt đầu từ 1.
for i, document in enumerate(results, start=1):
    # In tiêu đề cho từng kết quả.
    print(f'--- Kết quả {i} ---')
    # In nội dung đoạn PDF tìm được.
    print(document.page_content)


--- Kết quả 1 ---
sản và bồi thường thiệt hại. 
Chương XIII 
QUYỀN SỞ HỮU 
Mục 1. NỘI DUNG QUYỀN SỞ HỮU 
Tiểu mục 1. QUYỀN CHIẾM HỮU 
Điều 186. Quyền chiếm hữu của chủ sở hữu 
Chủ sở hữu được thực hiện mọi hành vi theo ý chí của mình để nắm giữ, chi phối tài sản của mình nhưng không 
được trái pháp luật, đạo đức xã hội. 
Điều 187. Quyền chiếm hữu của người được chủ sở hữu ủy quyền quản lý tài sản 
1. Người được chủ sở hữu ủy quyền quản lý tài sản thực hiện việc chiếm hữu tài sản đó trong phạm vi, theo cách
--- Kết quả 2 ---
năm 1980 và Hiến pháp năm 1992, Nhân dân Việt Nam xây dựng, thi hành và 
bảo vệ Hiến pháp này vì m ục tiêu dân giàu, nư ớc mạnh, dân ch ủ, công b ằng, 
văn minh.  
CHƯƠNG I 
CHẾ ĐỘ CHÍNH TRỊ 
Điều 1  
Nước Cộng hòa xã hội chủ nghĩa Việt Nam là m ột nước độc lập, có ch ủ 
quyền, thống nhất và toàn v ẹn lãnh th ổ, bao g ồm đất liền, hải đảo, vùng bi ển 
và vùng trời.  
Điều 2   
1. Nhà nư ớc C ộng hòa xã h ội ch ủ nghĩa Vi ệt Nam là nhà n ước pháp
--- Kết quả 3